# Step 4: Evaluation & Visualization
**TSU NSF AI Workshop 2026 — Federated Learning Lab**

---

## What are we doing in this notebook?

We now compare our two approaches side by side:

| Approach | Description |
|----------|-------------|
| **Centralized** | All data combined on one server — traditional AI training |
| **Federated** | Data stays on each client — only model weights are shared |

We will look at:
1. **Final test accuracy** — how well does each model classify skin lesions?
2. **Convergence behavior** — how does accuracy change over training time?
3. **Communication cost** — how much data travels over the network in FL?

**Prerequisite:** You must have run notebooks 1 and 3 first so the result files exist in your Google Drive.

## 🔧 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import matplotlib.pyplot as plt
import numpy as np

DRIVE_BASE  = "/content/drive/MyDrive/FederatedLearning"
RESULTS_DIR = os.path.join(DRIVE_BASE, "results")
PLOTS_DIR   = os.path.join(RESULTS_DIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

central_path  = os.path.join(RESULTS_DIR, "centralized_results.json")
federated_path = os.path.join(RESULTS_DIR, "federated_results.json")

for label, path in [("Centralized results", central_path),
                    ("Federated results",   federated_path)]:
    status = "✅ Found" if os.path.exists(path) else "❌ Missing!"
    print(f"  {label}: {status}")

## Load Results

In [ ]:
with open(central_path)  as f: central   = json.load(f)
with open(federated_path) as f: federated = json.load(f)

print(f"Centralized  — Epochs: {central['epochs']},  Final test accuracy: {central['test_acc']:.2%}")
print(f"Federated    — Rounds: {federated['num_rounds']}, Local epochs/round: {federated['local_epochs']}, Final test accuracy: {federated['test_acc']:.2%}")

## Part 1: Accuracy Comparison

Let's look at the final accuracy numbers first.

In [ ]:
print("=" * 50)
print("   ACCURACY COMPARISON")
print("=" * 50)
print(f"   Centralized (baseline) : {central['test_acc']:.2%}")
print(f"   Federated (2 clients)  : {federated['test_acc']:.2%}")

diff = federated['test_acc'] - central['test_acc']
sign = "+" if diff >= 0 else ""
print(f"\n   Accuracy gap           : {sign}{diff:.2%}")

if abs(diff) < 0.03:
    print("   → Great! Federated achieves similar accuracy despite privacy preservation.")
elif diff > 0:
    print("   → Federated even outperformed centralized!")
else:
    print("   → Expected: Non-IID data creates a gap. This is the core FL challenge.")
    print("     More rounds or less non-IID skew would narrow this gap.")

## Part 2: Training Curves

How does accuracy change over training time?
- **Left:** centralized — accuracy per epoch
- **Right:** federated — accuracy per communication round

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Centralized vs. Federated Learning — DermaMNIST", fontsize=14)

# Left: Centralized
ax = axes[0]
epochs = list(range(1, central["epochs"] + 1))
ax.plot(epochs, central["history"]["train_acc"], marker="o", label="Train Accuracy", color="steelblue")
ax.plot(epochs, central["history"]["val_acc"],   marker="s", label="Val Accuracy",   color="darkorange")
ax.set_title("Centralized Training")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
final_val = central["history"]["val_acc"][-1]
ax.annotate(f"{final_val:.2%}", xy=(epochs[-1], final_val),
            xytext=(-35, 8), textcoords="offset points", fontsize=9, color="darkorange")

# Right: Federated
ax = axes[1]
rounds = federated["history"]["round"]
accs   = federated["history"]["test_acc"]
ax.plot(rounds, accs, marker="o", color="seagreen", label="Global Test Accuracy")
ax.set_title("Federated Training (2 Clients, FedAvg)")
ax.set_xlabel("Communication Round")
ax.set_ylabel("Test Accuracy")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
ax.annotate(f"{accs[-1]:.2%}", xy=(rounds[-1], accs[-1]),
            xytext=(-35, 8), textcoords="offset points", fontsize=9, color="seagreen")

plt.tight_layout()
path = os.path.join(PLOTS_DIR, "accuracy_comparison.png")
plt.savefig(path, dpi=120)
print(f"Saved: {path}")
plt.show()

## Part 3: Final Accuracy Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
methods = ["Centralized", "Federated\n(2 clients)"]
accs    = [central["test_acc"], federated["test_acc"]]
bars    = ax.bar(methods, accs, color=["steelblue", "seagreen"], width=0.45, edgecolor="white")

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{acc:.2%}", ha="center", va="bottom", fontsize=13, fontweight="bold")

ax.set_ylim(0, 1.0)
ax.set_ylabel("Test Accuracy", fontsize=12)
ax.set_title("Final Test Accuracy: Centralized vs. Federated", fontsize=12)
ax.axhline(central["test_acc"], color="steelblue", linestyle="--", alpha=0.4, label="Centralized baseline")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
path = os.path.join(PLOTS_DIR, "final_accuracy_bar.png")
plt.savefig(path, dpi=120)
print(f"Saved: {path}")
plt.show()

## Part 4: Communication Cost Analysis

In federated learning, model weights are sent back and forth each round.
How much data is that — and how does accuracy improve as cost accumulates?

In [ ]:
MODEL_SIZE_MB = 2.0   # SimpleCNN ~500K params × 4 bytes ≈ 2 MB
n_rounds  = federated["num_rounds"]
n_clients = 2

print("=" * 50)
print("   COMMUNICATION COST")
print("=" * 50)
total_comm = n_rounds * n_clients * 2 * MODEL_SIZE_MB
print(f"   Rounds          : {n_rounds}")
print(f"   Clients         : {n_clients}")
print(f"   Model size      : {MODEL_SIZE_MB} MB")
print(f"   Total comm. cost: {total_comm:.1f} MB (send + receive)")
raw_data_mb = 7007 * 28 * 28 * 3 / 1e6
print(f"\n   Raw training data size : {raw_data_mb:.1f} MB")
print(f"   Patient data protected : {raw_data_mb:.1f} MB (never transmitted!)")

# Chart: accuracy vs cumulative communication cost
cumulative_costs = [r * n_clients * 2 * MODEL_SIZE_MB
                    for r in federated["history"]["round"]]
fed_accs = federated["history"]["test_acc"]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(cumulative_costs, fed_accs, marker="o", color="seagreen", label="Federated")
ax.axhline(central["test_acc"], color="steelblue", linestyle="--",
           label=f"Centralized ({central['test_acc']:.2%})")
for cost, acc, r in zip(cumulative_costs, fed_accs, federated["history"]["round"]):
    ax.annotate(f"R{r}", xy=(cost, acc), xytext=(4, 4),
                textcoords="offset points", fontsize=8)
ax.set_xlabel("Cumulative Communication Cost (MB)", fontsize=11)
ax.set_ylabel("Test Accuracy", fontsize=11)
ax.set_title("Accuracy vs. Communication Cost", fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
path = os.path.join(PLOTS_DIR, "convergence_vs_cost.png")
plt.savefig(path, dpi=120)
print(f"\nSaved: {path}")
plt.show()

## Key Takeaways

| Observation | What it means |
|-------------|---------------|
| Federated accuracy is close to centralized | FL preserves privacy without sacrificing much accuracy |
| Accuracy grows with more rounds | More communication → better model |
| Accuracy gap with non-IID data | Different patient populations make FL harder |
| Only model weights are transmitted | Patient images never leave the hospital |

**Privacy vs. Accuracy tradeoff:**
The small accuracy gap is the "cost" of privacy. In real healthcare settings, this tradeoff is absolutely worth it — patient data must stay local.

---

All plots have been saved to `results/plots/` in your Google Drive.

➡️ **Next step: `5_practice.ipynb`** — experiment with FL parameters and see what changes.